# Business Rule Validation Engine

> **Purpose**: Demonstrate the `rules.py` module — enforce domain-specific data quality rules and compute a business-rule quality score.

**All business logic lives in `../rules.py`. This notebook only imports and calls those functions.**

### Rules enforced
| # | Rule | Severity |
|---|------|----------|
| 1 | Required fields (Transaction_ID, Customer_ID, Product_Name) must not be null | High |
| 2 | Quantity and Price must not be negative | High |
| 3 | Transaction dates must not be in the future | Medium |
| 4 | Transaction_ID must be unique (no duplicates) | High |
| 5 | Product_Name must belong to the known category set | Low |

> **New in this version**: Each violation includes `count`, `percentage`, and `sample_indices` (max 10). A `rules_quality_score` is computed at the end.

> **Previous step**: `validation.ipynb` | **Next step**: `statistics.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from rules import (
    check_missing_required_fields,
    check_negative_values,
    check_future_dates,
    check_duplicate_ids,
    check_unknown_categories,
    compute_rules_quality_score,
    rules_summary,
    run_business_rules,
)

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)
print(f"Dataset loaded and cleaned: {df_clean.shape}")

## Rule 1 — Missing Required Fields

In [ ]:
missing_v = check_missing_required_fields(df_clean)
for v in missing_v:
    print(f"[{v['severity']:6}] {v['rule']}")
    print(f"         Count: {v['count']:,} | {v['percentage']}% of rows")
    print(f"         Sample indices: {v['sample_indices']}")

## Rule 2 — Negative Values (Quantity / Price)

In [ ]:
neg_v = check_negative_values(df_clean)
for v in neg_v:
    print(f"[{v['severity']:6}] {v['rule']}")
    print(f"         Count: {v['count']:,} | {v['percentage']}% of rows")
    print(f"         Sample indices: {v['sample_indices']}")

## Rule 3 — Future Dates

In [ ]:
future_v = check_future_dates(df_clean)
if future_v:
    for v in future_v:
        print(f"[{v['severity']:6}] {v['rule']} — {v['count']:,} rows ({v['percentage']}%)")
else:
    print("No future dates found — PASS")

## Rule 4 — Duplicate Transaction IDs

In [ ]:
dup_v = check_duplicate_ids(df_clean)
for v in dup_v:
    print(f"[{v['severity']:6}] {v['rule']} — {v['count']:,} rows ({v['percentage']}%)")
    print(f"         Sample indices: {v['sample_indices']}")

## Rule 5 — Unknown Product Categories

In [ ]:
cat_v = check_unknown_categories(df_clean)
for v in cat_v:
    print(f"[{v['severity']:6}] {v['rule']} — {v['count']:,} rows ({v['percentage']}%)")
    print(f"  Top unknown values: {list(v.get('unknown_values', {}).keys())[:10]}")

## Full Run + Quality Score

In [ ]:
violations = run_business_rules(df_clean)

# Compute the business-rule quality score
quality_score = compute_rules_quality_score(violations, total_rows=len(df_clean))

# Get a structured summary
summary = rules_summary(violations, total_rows=len(df_clean))

print("=== Business Rules Summary ===")
print(f"  Violation types          : {summary['total_violation_types']}")
print(f"  Total affected records   : {summary['total_affected_records']:,}")
print(f"  Rules Quality Score      : {summary['rules_quality_score']}/100")
print(f"  Severity breakdown       : {summary['severity_breakdown']}")

print("\n=== Violation Details ===")
for v in violations:
    print(f"  [{v['severity']:6}] {v['rule']:<48} {v['count']:>6,} rows ({v['percentage']:.1f}%)")

---
## Key Takeaways

- The `rules_quality_score` gives a single 0–100 number summarising business rule health
- `sample_indices` (max 10) replaces the old 50-item `affected_indices` — keeps reports compact
- Negative Quantity and Price values are a major issue (>30% of rows each) — likely a data generation artifact
- Unknown categories suggest the product catalog needs updating in `config.KNOWN_CATEGORIES`
- This result feeds directly into `scoring.py` to compute `rules_quality_score` in the overall dataset score